[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/A.%20Foundational%20Analytics%20%26%20Inventory%20Control%20%28The%20Language%20of%20Supply%20Chain%29/049.%20The%20Causal%20%26%20Regression-Based%20Forecasting%20Problem/P49-Tier-2_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 49. The Causal and Regression-Based Forecasting Problem

## Tier 2: The Classic Heuristic — Forward Stepwise Regression

This notebook builds a linear regression model by adding predictors one at a time. At each step, it tests every predictor not yet selected, adds the one with the strongest statistically significant improvement, and stops when no remaining predictor meets the chosen significance threshold. The procedure is a greedy heuristic: it is computationally economical and produces an interpretable sequence of models, but it does not guarantee the globally best subset.

### Goals, inputs, and outputs

- **Goal:** demonstrate how forward stepwise regression selects predictors for a synthetic port-throughput model.
- **Inputs:** a reproducible synthetic dataset, a candidate predictor matrix, and a significance threshold of 0.05.
- **Outputs:** the sequence of selected predictors, nested-model F-tests, in-sample fit and information-criterion summaries, and diagnostic visualizations.
- **Method:** forward selection using partial F-tests to compare nested ordinary least-squares (OLS) models. AIC and BIC are reported as descriptive model summaries; they do not determine which predictor is added.

### Scope and modeling assumptions

The dataset is simulated, and the relationships used to generate throughput are known by construction. The notebook treats those generating predictors as the reference set when illustrating selection; selecting a predictor does not, by itself, establish causality in real observational data. The regression assumes a linear conditional mean and uses classical nested-model F-tests, whose usual interpretation relies on suitable error assumptions, including independent, constant-variance, normally distributed errors. All fit metrics shown here are in-sample; there is no held-out test set or time-based validation. The procedure does not model nonlinear effects, interactions, correlated measurement errors, or changing relationships over time.

### Key assumptions of the heuristic

- Greedy, local choices can produce a useful model, even though they may miss the best subset considered globally.
- The partial F-test is a reasonable screening rule under its modeling assumptions. Repeated testing across candidate predictors can make unadjusted p-values optimistic.
- A smaller selected model can be easier to interpret, but forward selection does not guarantee the smallest or best possible model.
- Reducing the search can be computationally attractive when exhaustive subset enumeration is impractical.

### Procedure

1. Start with an intercept-only model.
2. For each predictor not yet selected, fit the candidate model that adds it to the current model.
3. Compare each candidate with the current model using a partial F-test, and identify the candidate with the largest F-statistic.
4. Add that predictor if its p-value is below the significance threshold; otherwise stop.
5. Summarize the selected model and inspect its fit statistics and selection trajectory.

### Synthetic example

The candidate predictors represent trade volume, GDP growth, fuel price, seasonal peak, ship density, two random noise variables, and competition. The response is generated from the first five variables plus random error. The simulation makes this generating structure available for an illustrative selection analysis; finite-sample noise means that a selection procedure is not guaranteed to recover it exactly.

### Mathematical model and selection rule

For a selected predictor set \(S\), the model predicts throughput as an intercept plus a weighted sum of the selected predictors. Let \(y_i\) be throughput for observation \(i\), and let \(x_{ij}\) be its value for candidate predictor \(j\). The OLS fit for a set \(S\) minimizes the residual sum of squares:

$$
\operatorname{SSE}(S) = \min_{\beta_0,\{\beta_j:j\in S\}} \sum_{i=1}^{n} \left(y_i - \beta_0 - \sum_{j\in S}\beta_j x_{ij}\right)^2.
$$

Here, \(n\) is the number of observations, \(\beta_0\) is the intercept, and \(\beta_j\) is the coefficient for predictor \(j\). The intercept-only model is the base case \(S_0=\varnothing\). At step \(t\), the remaining candidates are \(R_t=\{1,\ldots,p\}\setminus S_t\), where \(p\) is the number of candidate predictors. For each \(j\in R_t\), compare the current model with the model that adds \(j\):

$$
F_j = \frac{\left[\operatorname{SSE}(S_t)-\operatorname{SSE}(S_t\cup\{j\})\right]/q}{\operatorname{SSE}(S_t\cup\{j\})/\left[n-(|S_t|+q+1)\right]},
\qquad q=1.
$$

The numerator measures the reduction in residual error per added parameter; the denominator estimates residual variance in the larger model. The \(+1\) in the residual degrees of freedom counts the intercept. More generally, \(q\) is the number of added parameters. The implementation computes the corresponding degrees of freedom from the design-matrix column counts.

The selected candidate is \(j_t^*=\arg\max_{j\in R_t}F_j\). If its p-value is below the threshold \(\alpha\), update \(S_{t+1}=S_t\cup\{j_t^*\}\); otherwise stop and retain \(S_t\). In the code, `selected_features` represents \(S_t\), `remaining_features` represents \(R_t\), and `calculate_f_statistic` evaluates each candidate. Ties in the computed F-statistic are resolved by the order of the candidate columns.

For a model with \(k\) fitted regression coefficients, the notebook also reports the conventions

$$
\operatorname{AIC}=n\log(\operatorname{MSE})+2k,
\qquad
\operatorname{BIC}=n\log(\operatorname{MSE})+k\log(n),
\qquad
\operatorname{MSE}=\frac{\operatorname{SSE}}{n}.
$$

Here \(k\) counts the intercept and selected regression coefficients. Under these conventions, lower AIC or BIC indicates a preferable trade-off between in-sample error and model size when comparing models fit to the same response data. These values are reported for interpretation, not used by the stepwise selection rule.

At most \(p\) predictors can be added. If all candidates are tested at every step, the number of candidate-model tests is at most \(p+(p-1)+\cdots+1=p(p+1)/2\), compared with \(2^p-1\) nonempty subsets in exhaustive subset enumeration. This comparison counts model evaluations, not equal-cost operations, and the stepwise procedure may stop before reaching its upper bound. Greedy selection is path-dependent and is not guaranteed to find the best subset under any global criterion.

In [ ]:
# Import the tools used to generate data, fit nested OLS models, and inspect results.
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
# Fix the random seed so that the simulated example is reproducible.
np.random.seed(42)

n_samples = 100

# These five predictors contribute to throughput in the data-generating process.
trade_volume = np.random.normal(100, 15, n_samples)
gdp_growth = np.random.normal(2.5, 0.8, n_samples)
fuel_price = np.random.normal(85, 10, n_samples)
seasonal_peak = np.random.binomial(1, 0.3, n_samples)
ship_density = np.random.normal(25, 5, n_samples)

# These candidates are generated independently of the response-generating equation.
random_noise_1 = np.random.normal(0, 1, n_samples)
random_noise_2 = np.random.normal(0, 1, n_samples)
competition = np.random.normal(50, 10, n_samples)

# Add random error to the specified linear data-generating relationship.
true_throughput = (
    2000
    + 15 * trade_volume
    + 300 * gdp_growth
    + 8 * fuel_price
    + 250 * seasonal_peak
    + 20 * ship_density
    + np.random.normal(0, 200, n_samples)
)

data = pd.DataFrame(
    {
        "Trade_Volume": trade_volume,
        "GDP_Growth": gdp_growth,
        "Fuel_Price": fuel_price,
        "Seasonal_Peak": seasonal_peak,
        "Ship_Density": ship_density,
        "Random_Noise_1": random_noise_1,
        "Random_Noise_2": random_noise_2,
        "Competition": competition,
        "Throughput": true_throughput,
    }
)

print("Synthetic port-throughput dataset")
print(f"Shape: {data.shape}")
print("\nFirst five rows:")
print(data.head().round(2))
print("\nDescriptive statistics:")
print(data.describe().round(2))

### Comparing nested models and fitting OLS

A design matrix is the numeric table of predictor values supplied to a regression model. Each candidate matrix below includes a leading column of ones, so the intercept is represented explicitly. The reduced model is nested in the full model: it can be obtained by setting the added predictor's coefficient to zero. The partial F-test asks whether the reduction in residual error is large relative to the residual variation left in the larger model.

`calculate_f_statistic` implements this comparison. `fit_ols_model` fits a specified design matrix and returns in-sample predictions, coefficients, MSE, \(R^2\), AIC, and BIC. The regression is fit without an additional implicit intercept because the design matrix already contains the intercept column.

In [ ]:
def calculate_f_statistic(X_full, X_reduced, y):
    """Compare nested OLS models using a partial F-test.

    Both design matrices must include their intercept column. The full model
    must contain all columns in the reduced model plus one or more predictors.
    """
    X_full = np.asarray(X_full, dtype=float)
    X_reduced = np.asarray(X_reduced, dtype=float)
    y = np.asarray(y, dtype=float)

    if X_full.ndim != 2 or X_reduced.ndim != 2:
        raise ValueError("Both design matrices must be two-dimensional.")
    if len(y) != X_full.shape[0] or len(y) != X_reduced.shape[0]:
        raise ValueError("The response and design matrices must have matching rows.")
    if X_full.shape[1] <= X_reduced.shape[1]:
        raise ValueError("The full model must contain additional parameters.")

    n = len(y)
    df_full = n - X_full.shape[1]
    df_reduced = n - X_reduced.shape[1]
    df_num = df_reduced - df_full
    if df_full <= 0 or df_num <= 0:
        raise ValueError("The models must have positive residual degrees of freedom.")

    model_full = LinearRegression(fit_intercept=False).fit(X_full, y)
    model_reduced = LinearRegression(fit_intercept=False).fit(X_reduced, y)
    residuals_full = y - model_full.predict(X_full)
    residuals_reduced = y - model_reduced.predict(X_reduced)
    sse_full = np.sum(residuals_full**2)
    sse_reduced = np.sum(residuals_reduced**2)

    # Nested OLS fits cannot worsen when parameters are added; clipping only
    # absorbs a tiny negative difference caused by floating-point rounding.
    improvement = max(sse_reduced - sse_full, 0.0)
    if np.isclose(sse_full, 0.0):
        f_stat = np.inf if improvement > 0 else 0.0
    else:
        f_stat = (improvement / df_num) / (sse_full / df_full)
    p_value = stats.f.sf(f_stat, df_num, df_full)

    return f_stat, p_value


def fit_ols_model(X, y):
    """Fit OLS to a design matrix with an explicit intercept column.

    Return the fitted coefficients and in-sample fit summaries. The first
    coefficient corresponds to the leading column of ones.
    """
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    if X.ndim != 2 or len(y) != X.shape[0]:
        raise ValueError("The design matrix and response must have matching rows.")
    if X.shape[1] < 1:
        raise ValueError("The design matrix must include an intercept column.")

    model = LinearRegression(fit_intercept=False).fit(X, y)
    predictions = model.predict(X)
    mse = mean_squared_error(y, predictions)
    r2 = r2_score(y, predictions)

    n = len(y)
    k = X.shape[1]
    aic = n * np.log(mse) + 2 * k if mse > 0 else -np.inf
    bic = n * np.log(mse) + k * np.log(n) if mse > 0 else -np.inf

    return {
        "coefficients": model.coef_,
        "intercept": model.coef_[0],
        "mse": mse,
        "r2": r2,
        "aic": aic,
        "bic": bic,
        "predictions": predictions,
    }

### Forward stepwise regression

The algorithm maintains two complementary sets: predictors already selected and predictors still available for testing. After a candidate is added, the next round tests every remaining predictor conditional on the enlarged model. This repeated conditional testing is what makes the procedure path-dependent: an early choice changes the comparisons made later.

The code records the best candidate and its test statistic at each accepted step. It stops when the best remaining candidate fails the significance threshold. The supplied `feature_names` keep predictor labels aligned with the columns of `X`; the function does not rely on a global dataset.

In [ ]:
def forward_stepwise_regression(X, y, significance_level=0.05, feature_names=None):
    """Select predictors by forward stepwise partial F-tests.

    X contains candidate predictors without an intercept column. The function
    starts with an intercept-only model and adds the strongest significant
    candidate at each step.
    """
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    if X.ndim != 2 or len(y) != X.shape[0]:
        raise ValueError("The predictor matrix and response must have matching rows.")
    if not 0 < significance_level < 1:
        raise ValueError("significance_level must be between 0 and 1.")

    n_samples, n_predictors = X.shape
    if feature_names is None:
        feature_names = [f"Feature_{i}" for i in range(n_predictors)]
    if len(feature_names) != n_predictors:
        raise ValueError("feature_names must contain one name per predictor column.")

    selected_features = []
    remaining_features = list(range(n_predictors))
    step_results = []
    current_X = np.ones((n_samples, 1))

    print("Forward stepwise regression progress:")
    print("=" * 60)

    step = 0
    while remaining_features and step < n_predictors:
        best_f_stat = -np.inf
        best_feature = None
        best_p_value = 1.0

        for feature in remaining_features:
            candidate_X = np.column_stack([current_X, X[:, feature]])
            f_stat, p_value = calculate_f_statistic(candidate_X, current_X, y)

            # Strict comparison makes ties deterministic in column order.
            if f_stat > best_f_stat:
                best_f_stat = f_stat
                best_feature = feature
                best_p_value = p_value

        if best_feature is not None and best_p_value < significance_level:
            selected_features.append(best_feature)
            remaining_features.remove(best_feature)
            current_X = np.column_stack([current_X, X[:, best_feature]])
            model_results = fit_ols_model(current_X, y)

            step_result = {
                "step": step + 1,
                "feature_added": best_feature,
                "feature_name": feature_names[best_feature],
                "f_statistic": best_f_stat,
                "p_value": best_p_value,
                "selected_features": selected_features.copy(),
                "mse": model_results["mse"],
                "r2": model_results["r2"],
                "aic": model_results["aic"],
                "bic": model_results["bic"],
            }
            step_results.append(step_result)

            print(f"Step {step + 1}: Added {feature_names[best_feature]}")
            print(f"  F-statistic: {best_f_stat:.4f}")
            print(f"  p-value: {best_p_value:.6g}")
            print(f"  R-squared: {model_results['r2']:.4f}")
            print()
            step += 1
        else:
            print(
                "No remaining feature meets significance level "
                f"{significance_level}."
            )
            break

    final_model = fit_ols_model(current_X, y) if selected_features else None
    return {
        "selected_features": selected_features,
        "step_results": step_results,
        "final_model": final_model,
    }

### Prepare the predictor matrix and run the selection procedure

The response is separated from the candidate predictors so that no response values can accidentally enter the feature-selection process. The order of `predictor_columns` defines the column indices used throughout the algorithm and is passed explicitly to preserve the correct feature names.

In [ ]:
predictor_columns = [column for column in data.columns if column != "Throughput"]
X = data[predictor_columns].to_numpy()
y = data["Throughput"].to_numpy()

print(f"Predictor variables: {predictor_columns}")
print(f"Number of candidate predictors: {len(predictor_columns)}")
print(f"Sample size: {len(y)}")

stepwise_results = forward_stepwise_regression(
    X,
    y,
    significance_level=0.05,
    feature_names=predictor_columns,
)

### Interpret the selected model

The fitted intercept is the predicted response when all included predictors equal zero; whether that value is meaningful depends on the predictors' scales and context. Each slope is the model's estimated change in throughput for a one-unit increase in that predictor, holding the other selected predictors fixed. These are conditional regression associations, not automatic causal effects. MSE is expressed in squared response units, whereas \(R^2\) is a unitless measure of in-sample variation explained.

In [ ]:
print("\n" + "=" * 60)
print("FINAL MODEL SUMMARY")
print("=" * 60)

if stepwise_results["final_model"] is not None:
    final_model = stepwise_results["final_model"]
    selected_features = stepwise_results["selected_features"]

    selected_names = [predictor_columns[index] for index in selected_features]
    print(f"Selected features: {selected_names}")
    print(f"Intercept: {final_model['intercept']:.4f}")

    for coefficient_position, feature_index in enumerate(selected_features, start=1):
        feature_name = predictor_columns[feature_index]
        coefficient = final_model["coefficients"][coefficient_position]
        print(f"{feature_name}: {coefficient:.4f}")

    print("\nIn-sample model performance:")
    print(f"MSE: {final_model['mse']:.2f}")
    print(f"R-squared: {final_model['r2']:.4f}")
    print(f"AIC: {final_model['aic']:.2f}")
    print(f"BIC: {final_model['bic']:.2f}")
else:
    print("No predictors were selected; the procedure retained the intercept-only model.")

### Visualize the selection path

Each point represents a model after an accepted addition. The \(R^2\) and MSE panels show in-sample fit; the AIC and BIC panel adds a complexity penalty; and the final panel shows each added variable's partial F-statistic and p-value. For a one-parameter addition, the 5% critical F-value depends on the residual degrees of freedom, so the plotted reference line varies by step. These plots describe the selection path; they are not an independent validation of predictive performance.

In [ ]:
if stepwise_results["step_results"]:
    results = stepwise_results["step_results"]
    steps = [result["step"] for result in results]
    r2_values = [result["r2"] for result in results]
    mse_values = [result["mse"] for result in results]
    aic_values = [result["aic"] for result in results]
    bic_values = [result["bic"] for result in results]
    f_statistics = [result["f_statistic"] for result in results]
    p_values = [result["p_value"] for result in results]
    feature_names = [result["feature_name"] for result in results]

    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle("Forward Stepwise Regression Analysis", fontsize=16, fontweight="bold")

    axes[0, 0].plot(steps, r2_values, "o-", linewidth=2, markersize=8, color="blue")
    axes[0, 0].set_xlabel("Accepted step")
    axes[0, 0].set_ylabel("R-squared")
    axes[0, 0].set_title("In-Sample Fit (R-squared)")
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].set_ylim(0, 1)
    for step, r2_value, name in zip(steps, r2_values, feature_names):
        axes[0, 0].annotate(
            name,
            (step, r2_value),
            textcoords="offset points",
            xytext=(0, 10),
            ha="center",
            fontsize=8,
        )

    axes[0, 1].plot(steps, mse_values, "o-", linewidth=2, markersize=8, color="red")
    axes[0, 1].set_xlabel("Accepted step")
    axes[0, 1].set_ylabel("Mean squared error (squared response units)")
    axes[0, 1].set_title("In-Sample Error (MSE)")
    axes[0, 1].grid(True, alpha=0.3)

    axes[1, 0].plot(steps, aic_values, "o-", linewidth=2, markersize=8, label="AIC", color="green")
    axes[1, 0].plot(steps, bic_values, "s-", linewidth=2, markersize=8, label="BIC", color="orange")
    axes[1, 0].set_xlabel("Accepted step")
    axes[1, 0].set_ylabel("Information criterion (lower is preferred)")
    axes[1, 0].set_title("Model Selection Summaries")
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    ax_f = axes[1, 1]
    positions = np.arange(len(f_statistics))
    bars = ax_f.bar(
        positions,
        f_statistics,
        alpha=0.7,
        color="purple",
        edgecolor="black",
        label="Observed F-statistic",
    )
    ax_f.set_xlabel("Accepted step")
    ax_f.set_ylabel("Partial F-statistic")
    ax_f.set_title("Variable-Addition Tests")
    ax_f.set_xticks(positions)
    ax_f.set_xticklabels([f"Step {step}" for step in steps])
    ax_f.grid(True, alpha=0.3, axis="y")

    for bar, p_value in zip(bars, p_values):
        height = bar.get_height()
        ax_f.annotate(
            f"p={p_value:.3g}",
            (bar.get_x() + bar.get_width() / 2, height),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=8,
        )

    residual_degrees_of_freedom = [n_samples - (step + 1) for step in steps]
    f_critical_values = [
        stats.f.ppf(0.95, 1, degrees_of_freedom)
        for degrees_of_freedom in residual_degrees_of_freedom
    ]
    ax_f.plot(
        positions,
        f_critical_values,
        "r--",
        label="5% critical value (1 numerator df)",
    )
    ax_f.legend()

    fig.tight_layout()
    plt.show()
else:
    print("There are no accepted predictor additions to visualize.")

### Compare with the full predictor model

The full model includes every candidate predictor, while the stepwise model includes only the selected predictors. Their in-sample fit and information criteria help illustrate the trade-off between fit and model size. This is not a comparison with exhaustive search: the subset count below is a theoretical model-evaluation count, and no exhaustive search is run. The stepwise evaluation count is an upper bound on candidate F-tests if every predictor were added; early stopping can reduce the actual count.

In [ ]:
print("\n" + "=" * 60)
print("MODEL COMPARISON")
print("=" * 60)

X_full = np.column_stack([np.ones(len(y)), X])
full_model_results = fit_ols_model(X_full, y)

print(f"Full model ({len(predictor_columns)} predictors):")
print(f"  R-squared: {full_model_results['r2']:.4f}")
print(f"  MSE: {full_model_results['mse']:.2f}")
print(f"  AIC: {full_model_results['aic']:.2f}")
print(f"  BIC: {full_model_results['bic']:.2f}")

if stepwise_results["final_model"] is not None:
    stepwise_model = stepwise_results["final_model"]
    selected_count = len(stepwise_results["selected_features"])
    print(f"\nStepwise model ({selected_count} predictors):")
    print(f"  R-squared: {stepwise_model['r2']:.4f}")
    print(f"  MSE: {stepwise_model['mse']:.2f}")
    print(f"  AIC: {stepwise_model['aic']:.2f}")
    print(f"  BIC: {stepwise_model['bic']:.2f}")

    r2_loss = full_model_results["r2"] - stepwise_model["r2"]
    mse_increase = stepwise_model["mse"] - full_model_results["mse"]
    aic_improvement = full_model_results["aic"] - stepwise_model["aic"]
    bic_improvement = full_model_results["bic"] - stepwise_model["bic"]

    print("\nIn-sample trade-offs (positive information-criterion improvement favors stepwise):")
    print(f"  R-squared difference (full minus stepwise): {r2_loss:.4f}")
    print(f"  MSE difference (stepwise minus full): {mse_increase:.2f}")
    print(f"  AIC improvement (full minus stepwise): {aic_improvement:.2f}")
    print(f"  BIC improvement (full minus stepwise): {bic_improvement:.2f}")

    features_removed = len(predictor_columns) - selected_count
    reduction_percentage = features_removed / len(predictor_columns) * 100
    print("\nModel size:")
    print(
        f"  Predictors omitted: {features_removed}/{len(predictor_columns)} "
        f"({reduction_percentage:.1f}%)"
    )
    print("  A smaller model can be easier to communicate, but it is not necessarily more accurate out of sample.")

p = len(predictor_columns)
exhaustive_combinations = 2**p - 1
stepwise_evaluation_upper_bound = p * (p + 1) // 2

print("\nTheoretical model-evaluation counts:")
print(f"  Nonempty subsets in exhaustive enumeration: {exhaustive_combinations:,}")
print(
    "  Upper bound on candidate F-tests in forward selection: "
    f"{stepwise_evaluation_upper_bound:,}"
)
print("  These counts describe search size, not measured runtime.")
print("  Exhaustive subset enumeration grows exponentially with the number of predictors.")

### Assess selection against the known simulation structure

Because the response was generated from five specified predictors, the simulation provides a reference set for illustrating selection behavior. The labels below distinguish generating predictors from candidates excluded from the response-generating equation. A noise variable can still appear associated by chance, especially when many candidates are tested; conversely, a generating predictor can be missed. Precision, recall, and accuracy summarize agreement with this constructed reference set, not proof of causal discovery or generalization to new data.

In [ ]:
print("\n" + "=" * 60)
print("SELECTION AGAINST THE DATA-GENERATING REFERENCE SET")
print("=" * 60)

generating_predictors = [
    "Trade_Volume",
    "GDP_Growth",
    "Fuel_Price",
    "Seasonal_Peak",
    "Ship_Density",
]
non_generating_candidates = ["Random_Noise_1", "Random_Noise_2", "Competition"]
selected_names = [
    predictor_columns[index]
    for index in stepwise_results["selected_features"]
]

print("Predictors used to generate throughput:")
for variable in generating_predictors:
    status = "Selected" if variable in selected_names else "Not selected"
    print(f"  {variable}: {status}")

print("\nCandidates not used in the response-generating equation:")
for variable in non_generating_candidates:
    status = "Selected" if variable in selected_names else "Not selected"
    print(f"  {variable}: {status}")

true_positives = sum(variable in selected_names for variable in generating_predictors)
false_positives = sum(variable in selected_names for variable in non_generating_candidates)
false_negatives = sum(variable not in selected_names for variable in generating_predictors)
true_negatives = sum(variable not in selected_names for variable in non_generating_candidates)

precision = (
    true_positives / (true_positives + false_positives)
    if true_positives + false_positives > 0
    else 0.0
)
recall = (
    true_positives / (true_positives + false_negatives)
    if true_positives + false_negatives > 0
    else 0.0
)
accuracy = (true_positives + true_negatives) / (
    true_positives + false_positives + false_negatives + true_negatives
)

print("\nAgreement with the simulation reference set:")
print(f"  Precision: {precision:.3f}")
print(f"  Recall: {recall:.3f}")
print(f"  Accuracy: {accuracy:.3f}")
print("  These metrics describe this simulated sample and reference labeling only.")

### Why this tier is useful

Forward stepwise regression addresses the feature-selection problem when a model begins with many candidate predictors. It offers an explicit, inspectable selection path and requires at most a quadratic number of candidate F-tests if it continues until all predictors have been added. Exhaustive subset enumeration considers all combinations and grows exponentially in the number of predictors; the comparison here is about theoretical search size, not measured execution time.

### Strengths and limitations

**Strengths**

- Automates a sequence of conditional predictor comparisons.
- Often yields a smaller model that is easier to interpret than the full candidate model.
- Makes the selection decisions and their F-statistics visible at each step.
- Can be practical when exhaustive enumeration is too large.

**Limitations**

- Greedy selection can miss a useful combination when predictors are considered jointly.
- The path depends on earlier selections and can be sensitive to the significance threshold.
- Repeated unadjusted significance tests can inflate false-positive risk; conventional p-values after selection do not account for the full selection process.
- Correlated predictors can make individual entry decisions unstable.
- A good in-sample fit does not establish out-of-sample predictive performance or causality.

### When to use this tier

Treat this procedure as an interpretable exploratory or baseline method when there are multiple candidate predictors and a transparent selection path is useful. For consequential forecasting, assess predictive performance on data not used for selection, and consider whether the linear model and its error assumptions are appropriate. Any extensions should preserve the distinction between exploratory variable selection, predictive validation, and causal inference.